# NetShield — Phase 9: Controlled Cross-Dataset Domain Adaptation

## Research question

**Can unsupervised domain adaptation genuinely improve cross-dataset NIDS performance when the target evaluation data remain completely unseen during adaptation?**

This phase is a corrected and stricter follow-up to the initial CORAL experiment.

### Why this phase exists

The first CORAL experiment produced an interesting but asymmetric result:
- some F1 scores improved dramatically;
- false-positive rates also became very high;
- target evaluation data were used to estimate adaptation statistics.

Therefore, we now separate the target domain into:

**unlabeled adaptation subset → CORAL statistics**

and

**untouched evaluation subset → final metrics**

Target labels are never used for alignment or model selection.

### Controlled comparison

For both transfer directions:

1. Common semantic baseline
2. CORAL
3. CORAL + class weighting

Models:
- Logistic Regression
- Random Forest

Primary metrics:
- F1
- Macro-F1
- PR-AUC
- Precision
- Recall
- FPR
- FNR
- ROC-AUC

This directly advances the project's central objective of improving and understanding cross-dataset robustness.


In [ ]:
from pathlib import Path
import gc, json, warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

UNSW_DIR = ROOT / "data" / "raw" / "UNSW-NB15"
CICIDS_DIR = ROOT / "data" / "raw" / "CICIDS2017"
OUT = ROOT / "outputs" / "controlled_domain_adaptation"
TABLES = OUT / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Output:", TABLES)


In [ ]:
def find_file(root, filename):
    matches = list(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Missing: {filename} under {root}")
    return matches[0]

unsw_train = pd.read_csv(find_file(UNSW_DIR, "UNSW_NB15_training-set.csv"))
unsw_test = pd.read_csv(find_file(UNSW_DIR, "UNSW_NB15_testing-set.csv"))

unsw_train.columns = unsw_train.columns.astype(str).str.strip()
unsw_test.columns = unsw_test.columns.astype(str).str.strip()

unsw_train["label_binary"] = pd.to_numeric(
    unsw_train["label"], errors="coerce"
).astype("int8")
unsw_test["label_binary"] = pd.to_numeric(
    unsw_test["label"], errors="coerce"
).astype("int8")

print("UNSW train:", unsw_train.shape)
print("UNSW test :", unsw_test.shape)


In [ ]:
CICIDS_FILES = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infiltration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
]

def find_cicids(filename):
    matches = list(CICIDS_DIR.rglob(filename))
    if matches:
        return matches[0]
    # tolerate the common spelling in the local file
    if "Infiltration" in filename:
        alt = filename.replace("Infiltration", "Infilteration")
        matches = list(CICIDS_DIR.rglob(alt))
        if matches:
            return matches[0]
    raise FileNotFoundError(filename)

# Keep only the columns needed to construct the common representation + label.
# This greatly reduces RAM compared with loading the full 82-column frame.
CIC_KEEP = [
    "Flow Duration",
    "Total Fwd Packets",
    "Total Backward Packets",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Packets/s",
    "Fwd Packets/s",
    "Bwd Packets/s",
    "Average Packet Size",
    "Label",
]

parts = []
for fname in CICIDS_FILES:
    fp = find_cicids(fname)
    part = pd.read_csv(fp, usecols=lambda c: str(c).strip() in CIC_KEEP, low_memory=False)
    part.columns = part.columns.astype(str).str.strip()
    parts.append(part)
    print(fname, part.shape)

cicids = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

cicids["label_binary"] = (
    cicids["Label"].astype("string").str.strip().str.upper().ne("BENIGN").astype("int8")
)

print("CICIDS:", cicids.shape)


In [ ]:
COMMON = [
    "flow_duration_sec", "fwd_packets", "bwd_packets",
    "fwd_bytes", "bwd_bytes", "total_packets", "total_bytes",
    "packet_rate", "fwd_packet_rate", "bwd_packet_rate",
    "avg_packet_size"
]

def div(a, b):
    a = pd.to_numeric(a, errors="coerce").astype("float32")
    b = pd.to_numeric(b, errors="coerce").astype("float32")
    b = b.mask(b <= 0)
    return a / b

def common_unsw(df):
    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = pd.to_numeric(df["dur"], errors="coerce").astype("float32")
    x["fwd_packets"] = pd.to_numeric(df["spkts"], errors="coerce").astype("float32")
    x["bwd_packets"] = pd.to_numeric(df["dpkts"], errors="coerce").astype("float32")
    x["fwd_bytes"] = pd.to_numeric(df["sbytes"], errors="coerce").astype("float32")
    x["bwd_bytes"] = pd.to_numeric(df["dbytes"], errors="coerce").astype("float32")
    x["total_packets"] = x["fwd_packets"] + x["bwd_packets"]
    x["total_bytes"] = x["fwd_bytes"] + x["bwd_bytes"]
    x["packet_rate"] = div(x["total_packets"], x["flow_duration_sec"])
    x["fwd_packet_rate"] = div(x["fwd_packets"], x["flow_duration_sec"])
    x["bwd_packet_rate"] = div(x["bwd_packets"], x["flow_duration_sec"])
    x["avg_packet_size"] = div(x["total_bytes"], x["total_packets"])
    return x.replace([np.inf, -np.inf], np.nan).astype("float32")

def common_cicids(df):
    x = pd.DataFrame(index=df.index)
    x["flow_duration_sec"] = (
        pd.to_numeric(df["Flow Duration"], errors="coerce").astype("float32") / np.float32(1_000_000)
    )
    x["fwd_packets"] = pd.to_numeric(df["Total Fwd Packets"], errors="coerce").astype("float32")
    x["bwd_packets"] = pd.to_numeric(df["Total Backward Packets"], errors="coerce").astype("float32")
    x["fwd_bytes"] = pd.to_numeric(df["Total Length of Fwd Packets"], errors="coerce").astype("float32")
    x["bwd_bytes"] = pd.to_numeric(df["Total Length of Bwd Packets"], errors="coerce").astype("float32")
    x["total_packets"] = x["fwd_packets"] + x["bwd_packets"]
    x["total_bytes"] = x["fwd_bytes"] + x["bwd_bytes"]
    x["packet_rate"] = pd.to_numeric(df["Flow Packets/s"], errors="coerce").astype("float32")
    x["fwd_packet_rate"] = pd.to_numeric(df["Fwd Packets/s"], errors="coerce").astype("float32")
    x["bwd_packet_rate"] = pd.to_numeric(df["Bwd Packets/s"], errors="coerce").astype("float32")
    x["avg_packet_size"] = pd.to_numeric(df["Average Packet Size"], errors="coerce").astype("float32")
    return x.replace([np.inf, -np.inf], np.nan).astype("float32")

Utr = common_unsw(unsw_train)
Ute = common_unsw(unsw_test)
C = common_cicids(cicids)

yu_tr = unsw_train["label_binary"].to_numpy()
yu_te = unsw_test["label_binary"].to_numpy()
yc = cicids["label_binary"].to_numpy()

print("UNSW common:", Utr.shape, Ute.shape)
print("CICIDS common:", C.shape)


## 1. Create target adaptation/evaluation partitions

The target labels are **not used** to construct these partitions.

For each direction:
- 20% of target rows are an unlabeled adaptation pool;
- 80% are the final evaluation set.

The split is purely random and reproducible.

This means CORAL sees target **feature values only**, while the evaluation labels remain untouched until the final scoring step.


In [ ]:
# UNSW is the target in CICIDS → UNSW.
# We split the official UNSW test set without looking at its labels.
unsw_target_idx = np.arange(len(Ute))
unsw_adapt_idx, unsw_eval_idx = train_test_split(
    unsw_target_idx, test_size=0.80, random_state=RANDOM_STATE, shuffle=True
)

# CICIDS is the target in UNSW → CICIDS.
# Random sampling is label-blind; labels are retained only for later evaluation.
cicids_target_idx = np.arange(len(C))
cicids_adapt_idx, cicids_eval_idx = train_test_split(
    cicids_target_idx, test_size=0.80, random_state=RANDOM_STATE, shuffle=True
)

print("UNSW target adaptation:", len(unsw_adapt_idx))
print("UNSW target evaluation:", len(unsw_eval_idx))
print("CICIDS target adaptation:", len(cicids_adapt_idx))
print("CICIDS target evaluation:", len(cicids_eval_idx))

partition_audit = pd.DataFrame([
    {"direction":"CICIDS_to_UNSW","partition":"adaptation","rows":len(unsw_adapt_idx)},
    {"direction":"CICIDS_to_UNSW","partition":"evaluation","rows":len(unsw_eval_idx)},
    {"direction":"UNSW_to_CICIDS","partition":"adaptation","rows":len(cicids_adapt_idx)},
    {"direction":"UNSW_to_CICIDS","partition":"evaluation","rows":len(cicids_eval_idx)},
])
partition_audit.to_csv(TABLES/"01_target_partition_audit.csv", index=False)
display(partition_audit)


## 2. Source-only preprocessing

The source scaler is fitted **only on source training data**.

Missing values are median-imputed using source training medians.

The target adaptation/evaluation data are transformed using those source-fitted preprocessing parameters.

No target labels are involved.


In [ ]:
def source_prepare(X_source, X_target_adapt, X_target_eval):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    Xs = imputer.fit_transform(X_source)
    Xa = imputer.transform(X_target_adapt)
    Xe = imputer.transform(X_target_eval)

    Xs = scaler.fit_transform(Xs).astype("float64")
    Xa = scaler.transform(Xa).astype("float64")
    Xe = scaler.transform(Xe).astype("float64")
    return Xs, Xa, Xe, imputer, scaler


In [ ]:
# Prepare both transfer directions.
# Direction 1: UNSW -> CICIDS
X_u2c_src, X_u2c_adapt, X_u2c_eval, imp_u2c, sc_u2c = source_prepare(
    Utr,
    C.iloc[cicids_adapt_idx],
    C.iloc[cicids_eval_idx]
)

# Direction 2: CICIDS -> UNSW
# To keep RAM manageable, use a reproducible 600k-row labeled source sample.
# Sampling is allowed because CICIDS is the source here and source labels are available.
C_SOURCE_MAX = 600_000
if len(C) > C_SOURCE_MAX:
    c_source_idx = rng.choice(len(C), size=C_SOURCE_MAX, replace=False)
else:
    c_source_idx = np.arange(len(C))

X_c2u_src, X_c2u_adapt, X_c2u_eval, imp_c2u, sc_c2u = source_prepare(
    C.iloc[c_source_idx],
    Ute.iloc[unsw_adapt_idx],
    Ute.iloc[unsw_eval_idx]
)

yc_source = yc[c_source_idx]
yu_target_eval = yu_te[unsw_eval_idx]
yc_target_eval = yc[cicids_eval_idx]

print("UNSW -> CICIDS source:", X_u2c_src.shape)
print("UNSW -> CICIDS target eval:", X_u2c_eval.shape)
print("CICIDS -> UNSW source:", X_c2u_src.shape)
print("CICIDS -> UNSW target eval:", X_c2u_eval.shape)


In [ ]:
def cov_sqrt(A, power, eps=1e-5):
    vals, vecs = np.linalg.eigh(A)
    vals = np.clip(vals, eps, None)
    return (vecs * (vals ** power)) @ vecs.T

def fit_coral(X_source, X_target_adapt):
    ms = X_source.mean(axis=0)
    mt = X_target_adapt.mean(axis=0)

    Cs = np.cov(X_source - ms, rowvar=False) + 1e-4*np.eye(X_source.shape[1])
    Ct = np.cov(X_target_adapt - mt, rowvar=False) + 1e-4*np.eye(X_target_adapt.shape[1])

    A = cov_sqrt(Cs, -0.5) @ cov_sqrt(Ct, 0.5)

    return {
        "source_mean": ms,
        "target_mean": mt,
        "A": A
    }

def transform_source(X, coral):
    return (X - coral["source_mean"]) @ coral["A"] + coral["target_mean"]

coral_u2c = fit_coral(X_u2c_src, X_u2c_adapt)
coral_c2u = fit_coral(X_c2u_src, X_c2u_adapt)

X_u2c_src_coral = transform_source(X_u2c_src, coral_u2c)
X_c2u_src_coral = transform_source(X_c2u_src, coral_c2u)

print("CORAL transformations fitted using unlabeled target adaptation pools only.")


In [ ]:
MODELS = {
    "Logistic Regression": lambda weighted=False: Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced" if weighted else None,
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest": lambda weighted=False: Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced" if weighted else None,
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ])
}

def score_model(model, Xtr, ytr, Xte, yte, direction, method, model_name):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    prob = model.predict_proba(Xte)[:, 1]

    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0,1]).ravel()

    return {
        "direction": direction,
        "method": method,
        "model": model_name,
        "precision": precision_score(yte, pred, zero_division=0),
        "recall": recall_score(yte, pred, zero_division=0),
        "f1": f1_score(yte, pred, zero_division=0),
        "macro_f1": f1_score(yte, pred, average="macro", zero_division=0),
        "pr_auc": average_precision_score(yte, prob),
        "roc_auc": roc_auc_score(yte, prob),
        "fpr": fp/(fp+tn) if (fp+tn) else np.nan,
        "fnr": fn/(fn+tp) if (fn+tp) else np.nan,
        "tn": tn, "fp": fp, "fn": fn, "tp": tp
    }

results = []


## 3. UNSW → CICIDS

The CICIDS adaptation pool is used only for CORAL statistics. Its labels are not used. The CICIDS evaluation pool is untouched until scoring.

In [ ]:
for model_name, factory in MODELS.items():
    # Baseline
    results.append(score_model(
        factory(False),
        X_u2c_src, yu_tr,
        X_u2c_eval, yc_target_eval,
        "UNSW_to_CICIDS", "common_baseline", model_name
    ))

    # CORAL
    results.append(score_model(
        factory(False),
        X_u2c_src_coral, yu_tr,
        X_u2c_eval, yc_target_eval,
        "UNSW_to_CICIDS", "CORAL", model_name
    ))

    # CORAL + class weighting
    results.append(score_model(
        factory(True),
        X_u2c_src_coral, yu_tr,
        X_u2c_eval, yc_target_eval,
        "UNSW_to_CICIDS", "CORAL_class_weighted", model_name
    ))

u2c_results = pd.DataFrame(results)
display(u2c_results.sort_values("f1", ascending=False))


## 4. CICIDS → UNSW

The UNSW adaptation pool is used only for CORAL statistics. The remaining UNSW target rows are the final evaluation set.

In [ ]:
for model_name, factory in MODELS.items():
    # Baseline
    results.append(score_model(
        factory(False),
        X_c2u_src, yc_source,
        X_c2u_eval, yu_target_eval,
        "CICIDS_to_UNSW", "common_baseline", model_name
    ))

    # CORAL
    results.append(score_model(
        factory(False),
        X_c2u_src_coral, yc_source,
        X_c2u_eval, yu_target_eval,
        "CICIDS_to_UNSW", "CORAL", model_name
    ))

    # CORAL + class weighting
    results.append(score_model(
        factory(True),
        X_c2u_src_coral, yc_source,
        X_c2u_eval, yu_target_eval,
        "CICIDS_to_UNSW", "CORAL_class_weighted", model_name
    ))

all_results = pd.DataFrame(results)
display(all_results.sort_values(["direction","f1"], ascending=[True,False]))


In [ ]:
baseline = all_results[
    all_results["method"] == "common_baseline"
][[
    "direction","model","f1","macro_f1","pr_auc","roc_auc","fpr","fnr"
]].rename(columns={
    "f1":"baseline_f1",
    "macro_f1":"baseline_macro_f1",
    "pr_auc":"baseline_pr_auc",
    "roc_auc":"baseline_roc_auc",
    "fpr":"baseline_fpr",
    "fnr":"baseline_fnr"
})

comparison = all_results.merge(baseline, on=["direction","model"], how="left")
comparison["delta_f1"] = comparison["f1"] - comparison["baseline_f1"]
comparison["delta_macro_f1"] = comparison["macro_f1"] - comparison["baseline_macro_f1"]
comparison["delta_pr_auc"] = comparison["pr_auc"] - comparison["baseline_pr_auc"]
comparison["delta_roc_auc"] = comparison["roc_auc"] - comparison["baseline_roc_auc"]
comparison["delta_fpr"] = comparison["fpr"] - comparison["baseline_fpr"]
comparison["delta_fnr"] = comparison["fnr"] - comparison["baseline_fnr"]

display(comparison.sort_values(["direction","f1"], ascending=[True,False]))

all_results.to_csv(TABLES/"02_controlled_coral_results.csv", index=False)
comparison.to_csv(TABLES/"03_controlled_coral_improvement.csv", index=False)


## 5. Alignment diagnostics

We measure covariance mismatch **after source-only standardization**, so the diagnostic is not dominated by raw units.

This is a diagnostic, not a performance metric.


In [ ]:
def covariance_distance(X, Y):
    Cs = np.cov(X, rowvar=False)
    Ct = np.cov(Y, rowvar=False)
    return float(np.linalg.norm(Cs - Ct, ord="fro"))

alignment_diag = pd.DataFrame([
    {
        "direction":"UNSW_to_CICIDS",
        "covariance_distance_before": covariance_distance(X_u2c_src, X_u2c_adapt),
        "covariance_distance_after": covariance_distance(X_u2c_src_coral, X_u2c_adapt)
    },
    {
        "direction":"CICIDS_to_UNSW",
        "covariance_distance_before": covariance_distance(X_c2u_src, X_c2u_adapt),
        "covariance_distance_after": covariance_distance(X_c2u_src_coral, X_c2u_adapt)
    }
])

alignment_diag["absolute_change"] = (
    alignment_diag["covariance_distance_after"] -
    alignment_diag["covariance_distance_before"]
)

alignment_diag["relative_change_pct"] = (
    alignment_diag["absolute_change"] /
    alignment_diag["covariance_distance_before"] * 100
)

display(alignment_diag)
alignment_diag.to_csv(TABLES/"04_standardized_covariance_alignment.csv", index=False)


## 6. Main research decision table

A useful adaptation method should not be judged by F1 alone.

We flag a configuration as **promising** only when:
- F1 improves over the common baseline;
- PR-AUC does not collapse;
- FPR does not explode;
- the covariance mismatch is reduced.

This prevents us from declaring success simply because the classifier predicts more attacks.


In [ ]:
decision = comparison.copy()

decision["f1_improved"] = decision["delta_f1"] > 0
decision["pr_auc_not_worse"] = decision["delta_pr_auc"] >= -0.02
decision["fpr_not_exploded"] = decision["delta_fpr"] <= 0.10
decision["alignment_reduced"] = decision["direction"].map({
    "UNSW_to_CICIDS": alignment_diag.loc[
        alignment_diag.direction=="UNSW_to_CICIDS","relative_change_pct"
    ].iloc[0] < 0,
    "CICIDS_to_UNSW": alignment_diag.loc[
        alignment_diag.direction=="CICIDS_to_UNSW","relative_change_pct"
    ].iloc[0] < 0
})

decision["promising_under_rule"] = (
    decision["f1_improved"] &
    decision["pr_auc_not_worse"] &
    decision["fpr_not_exploded"] &
    decision["alignment_reduced"]
)

display(decision[[
    "direction","method","model","f1","delta_f1",
    "pr_auc","delta_pr_auc","fpr","delta_fpr",
    "alignment_reduced","promising_under_rule"
]].sort_values(["direction","f1"], ascending=[True,False]))

decision.to_csv(TABLES/"05_adaptation_decision_table.csv", index=False)


In [ ]:
manifest = {
    "phase": "09_controlled_domain_adaptation",
    "research_question": "Can unsupervised domain adaptation genuinely improve cross-dataset NIDS performance?",
    "method": "CORAL",
    "common_features": COMMON,
    "directions": ["UNSW_to_CICIDS", "CICIDS_to_UNSW"],
    "methods": ["common_baseline", "CORAL", "CORAL_class_weighted"],
    "models": ["Logistic Regression", "Random Forest"],
    "target_adaptation_fraction": 0.20,
    "target_evaluation_fraction": 0.80,
    "target_labels_used_for_alignment": False,
    "target_labels_used_for_model_selection": False,
    "cicids_source_cap": C_SOURCE_MAX,
    "random_state": RANDOM_STATE,
    "important_note": "This phase is the corrected follow-up to the initial CORAL experiment. Target adaptation and final evaluation rows are disjoint."
}

with open(OUT/"phase9_manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2)

print("PHASE 9 COMPLETE")
print("Results:", TABLES)
